In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### MoE Layer
- Multiple Experts (FFNs)
- Router that decides which expert is best suited for each token

```
top_k = 2
num_experts = 3
tokens = ["ferrari", "is", "the", "greatest", "f1", "team"]
```

| Token | Top-2 Experts | Weights |
| :--- | :--- | :--- |
| ferrari | [0, 2] | [0.7, 0.3] |
| is | [1, 2] | [0.6, 0.4] |
| the | [1, 0] | [0.8, 0.2] |
| greatest | [2, 0] | [0.9, 0.1] |
| f1 | [0, 2] | [0.5, 0.5] |
| team | [2, 1] | [0.85, 0.15] |


For "ferrari":
- Experts chosen: [0, 2]
- Weights: [0.7, 0.3]

The final representation of "ferrari" is a weighted combination of multiple experts (2 experts in this example): `output = 0.7 * expert_0_output + 0.3 * expert_2_output`

In [2]:
class MoELayer(nn.Module):
    def __init__(self, embedding_dim, num_experts=8, top_k=3):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        # Each expert is a small FFN
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embedding_dim, embedding_dim * 4),
                nn.GELU(),
                nn.Linear(embedding_dim * 4, embedding_dim)
            ) for _ in range(num_experts)
        ])

        # Router: Learns which expert to send a token to
        self.router = nn.Linear(embedding_dim, num_experts)

    def forward(self, x):
        """
        x shape: [B, T, D]
        """
        B, T, D = x.shape
        x_flat = x.view(-1, D) # [B * T, D]

        # Get routing weights
        router_logits = self.router(x_flat)
        weights = F.softmax(router_logits, dim=-1)

        # Select top-k experts
        topk_weights, topk_indices = torch.topk(weights, self.top_k, dim=-1)
        
        # Normalize weights
        topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)

        output = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            
            # Find token assigned to this expert
            mask = (topk_indices == i).any(dim=-1)
            if mask.any():
                expert_mask = (topk_indices == i)
                token_indices = mask.nonzero(as_tuple=True)[0]
                expert_output = expert(x_flat[token_indices])

                for k in range(self.top_k):
                    k_mask = (topk_indices[token_indices, k] == i)
                    if k_mask.any():
                        output[token_indices[k_mask]] += (
                            topk_weights[token_indices[k_mask], k].unsqueeze(-1) * expert_output[k_mask]
                        )

        return output.view(B, T, D)

In [3]:
class GemmaBlock(nn.Module):

    def __init__(self, embedding_dim, vocabulary_size):
        super().__init__()
        self.layer_embedding = nn.Embedding(vocabulary_size, embedding_dim) # Per-Layer Embedding
        self.attn = nn.MultiheadAttention(embedding_dim, num_heads=8, batch_first=True)
        self.moe = MoELayer(embedding_dim)
        self.post_attn_norm = nn.LayerNorm(embedding_dim)
        self.post_moe_norm = nn.LayerNorm(embedding_dim)

    def forward(self, x, input_ids, mask=None):
        # PLE injection
        x = x + self.layer_embedding(input_ids)

        # Attention and norm
        attn_out, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.post_attn_norm(x + attn_out)

        # MoE and norm
        moe_out, gate_weights = self.moe(x)
        x = self.post_moe_norm(x + moe_out)

        return x, gate_weights

class Gemma(nn.Module):

    def __init__(self, embedding_dim=256, vocabulary_size=10000):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.token_embedding = nn.Embedding(vocabulary_size, embedding_dim)
        self.vision_projection = nn.Linear(1152, embedding_dim) # From Vision Encoder (SigLIP)
        self.audio_projection = nn.Linear(1024, embedding_dim) # From Audio Encoder (Conformer)

        self.layers = nn.ModuleList([
            GemmaBlock(embedding_dim, vocabulary_size) for _ in range(2)
        ])
        self.head = nn.Linear(embedding_dim, vocabulary_size)

    def forward(self, text_ids, vision_feats, audio_feats):
        # Project text, vision, audio to same latent space
        v_emedding = self.vision_projection(vision_feats) # [B, Tv, D]
        a_emedding = self.audio_projection(audio_feats) # [B, Ta, D]
        t_emedding = self.token_embedding(text_ids) # [B, Tt, D]

        # Concatenate: [Vision | Audio | Text]
        sequence = torch.cat([v_emedding, a_emedding, t_emedding], dim=1)

        # Create a dummy ids for the PLE (Vision/Audio positions get 0 index)
        vision_audio_size = v_emedding.size(1) + a_emedding.size(1) # Tv + Ta
        dummy_ids = torch.zeros(text_ids.size(0), vision_audio_size, dtype=torch.long, device=text_ids.device)
        input_ids = torch.cat([dummy_ids, text_ids], dim=1)

        # Apply causal mask
        seq_size = sequence.size(1) # [Tv + Ta + Tt]
        mask = torch.triu(torch.ones(seq_size, seq_size, device=text_ids.device), 1).bool()

        gate_weights = []
        x = sequence
        for layer in self.layers:
            x, gates = layer(x, input_ids, mask=mask)
            gate_weights.append(gates)

        return self.head(x), gate_weights
        

In [4]:
def load_balancing_loss(gate_weights_list):
    """
    This auxiliary or regularization loss is to ensure that router does not ignore experts
    """
    loss = 0
    for weights in gate_weights_list:
        mean_weights = weights.mean(dim=0)
        loss += torch.sum(mean_weights * torch.log(mean_weights.clamp(min=1e-6)))
    return loss

# Training
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Gemma().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# Mock data dimensions
vision_size = 8
audio_size = 12
text_size = 10

print("Starting training: ")
for epoch in range(5):
    for batch in range(4):
        
        # Create Mock inputs
        vision_feats = torch.randn(2, vision_size, 1152).to(device)
        audio_feats = torch.randn(2, audio_size, 1024).to(device)
        text_ids = torch.randint(1, 10_000, (2, text_size)).to(device)

        # Create labels with masking (-100 for non-text)
        vision_mask = torch.full((2, vision_size), -100).to(device)
        audio_mask = torch.full((2, audio_size), -100).to(device)
        labels = torch.cat([vision_mask, audio_mask, text_ids], dim=1)

        # Forward
        logits, gate_weights = model(text_ids, vision_feats, audio_feats)

        # Shifted loss: Align logit[i] w/ label[i+1]
        shifted_logits = logits[:, :-1, :].contiguous()
        shifted_labels = labels[:, 1:].contiguous()

        # Compute loss
        ce_loss = criterion(shifted_logits.view(-1, 10_000), shifted_labels.view(-1))
        aux_loss = load_balancing_loss(gate_weights)
        total_loss = ce_loss + 0.01 * aux_loss

        # Backward
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        optimizer.zero_grad()

        print(f"Epoch {epoch} | Batch {batch + 1} | Loss: {total_loss.item():.4f}")

print("Training Complete!")

Starting training: 
Epoch 0 | Batch 1 | Loss: 10.9484
Epoch 0 | Batch 2 | Loss: 10.3774
Epoch 0 | Batch 3 | Loss: 9.8630
Epoch 0 | Batch 4 | Loss: 9.6756
Epoch 1 | Batch 1 | Loss: 9.3577
Epoch 1 | Batch 2 | Loss: 9.0170
Epoch 1 | Batch 3 | Loss: 8.7745
Epoch 1 | Batch 4 | Loss: 8.7802
Epoch 2 | Batch 1 | Loss: 8.6130
Epoch 2 | Batch 2 | Loss: 8.4453
Epoch 2 | Batch 3 | Loss: 8.4493
Epoch 2 | Batch 4 | Loss: 8.2992
Epoch 3 | Batch 1 | Loss: 8.3314
Epoch 3 | Batch 2 | Loss: 8.1972
Epoch 3 | Batch 3 | Loss: 8.2360
Epoch 3 | Batch 4 | Loss: 8.1828
Epoch 4 | Batch 1 | Loss: 7.9967
Epoch 4 | Batch 2 | Loss: 8.0073
Epoch 4 | Batch 3 | Loss: 8.0200
Epoch 4 | Batch 4 | Loss: 7.8213
Training Complete!
